# **Static Deep Learning for S. aureus on DRIAMS set A with the longskip architecture**

## **Import the necessary libraries**

In [1]:
import os

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # must be set before torch is imported
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"


In [2]:
#utilities
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from collections import defaultdict
import copy
import re
import json
import umap
import shap
from tqdm import tqdm


# data splitting, metrics calculation
from skmultilearn.model_selection import IterativeStratification
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    balanced_accuracy_score,
    hamming_loss,
    confusion_matrix
)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import Normalizer


# model training
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.nn import BCELoss
from torch.optim.lr_scheduler import ReduceLROnPlateau

2025-09-28 20:17:26.247897: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759090646.492118      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759090646.562401      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## **Seed for reproducibility**

In [3]:
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    torch.manual_seed(seed)
    torch.Generator().manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    torch.set_float32_matmul_precision('high')

SEED = 3
seed_everything(3)

# **Utility Functions**

## **Function to make specific specie-antibiotic datasets**
This function reads a CSV file containing MALDI-TOF spectra and metadata, extracts numeric features (spectral bins), and filters the data for a specific bacterial species and selected antibiotics. It performs the following:

* Selects only rows corresponding to the target species.
* Filters out antibiotic columns that are not available or contain only NaNs.
* Combines spectral features with susceptibility labels for valid antibiotics.
* Removes rows with invalid or missing labels (only 'R', 'S', or 'I' are retained).

Returns a cleaned DataFrame ready for downstream AMR modeling.

In [4]:
def generate_driam_sets(specie, antibiotics, file_path, output_path = ""):
    """
    Extracts and prepares MALDI-TOF mass spectra and antibiotic resistance labels 
    for a given bacterial species from a DRIAM CSV file.

    Args:
        specie (str): Target bacterial species to extract.
        antibiotics (List[str]): List of antibiotic names to extract resistance labels for.
        file_path (str): Path to the input CSV file containing the DRIAM dataset.
        output_path (str, optional): Directory to save output files (not used here). Defaults to "".

    Returns:
        pd.DataFrame or None: Trimmed dataframe containing mass spectra and labels for the species.
                              Returns None if no data is found or antibiotics are missing.
    """
    print("Processing file at", file_path)
    bacteria = pd.read_csv(file_path)
    #Dropong all column with atleast one non-numeric character to Extract the feature columns i.e the bins
    malditof = bacteria[bacteria.columns.drop(list(bacteria.filter(regex='[^0-9]')))]

    print(f"\nProcessing species: {specie} in file: {file_path}")
    # We extract the species under study
    filtered_bacteria = bacteria[bacteria["species"] == specie]

    # Skip gracefully if no species are found in the file
    if filtered_bacteria.empty:
        print(f"Skipping {specie} in {file_path}: No rows found for species.")
        return

    # We'll only use those antibiotics in the given antibiotics which are actually present for that specie in dataframe
    available_antibiotics = [ab for ab in antibiotics if ab in filtered_bacteria.columns]

    # if no antibiotic is available in file, then skip gracefully
    if not available_antibiotics:
        print(f"Skipping {specie} in {file_path}: None of the specified antibiotics found.")
        return


    # If the label column cells are empty, replace them with the NaN
    filtered_bacteria[available_antibiotics] = filtered_bacteria[available_antibiotics].replace('', np.nan)

    # Drop columns that are completely NaN
    filtered_bacteria.dropna(axis=1, how='all', inplace=True)

    # Again get the available antibiotics in updated filtered dataframe where we dropped NaN columns
    available_antibiotics = [ab for ab in available_antibiotics if ab in filtered_bacteria.columns]

    # Making the new dataframes by combining mass spectra and antibiotic columns
    trimmed_bac = pd.DataFrame(
        np.column_stack([malditof.loc[filtered_bacteria.index], filtered_bacteria[available_antibiotics]]),
        columns=list(malditof.columns) + available_antibiotics
        )

    # Drop rows where any antibiotic label is NOT 'R' or 'S'
    for ab in antibiotics:
        trimmed_bac = trimmed_bac[trimmed_bac[ab].isin(['R', 'S', 'I'])]

    # Also drop rows with NaNs in antibiotic columns
    trimmed_bac = trimmed_bac.dropna(subset=antibiotics)

    # Proceed with analysis
    return trimmed_bac


## **Multi-label Stratified Train-Test Split**

This function performs a train-test split while preserving label distributions across multiple antibiotic resistance labels using `IterativeStratification`. This is crucial for multi-label classification problems, where standard stratification is not sufficient.

The number of splits is derived from the test size, and the function returns indices for training and testing data.

In [5]:
# Function to do train/test split stratified multi-label
def multilabel_train_test_split(X, y, test_size=0.2):
    """
    Performs stratified multi-label train/test split using iterative stratification.

    Args:
        X (np.ndarray): Input feature matrix.
        y (np.ndarray): Binary multi-label matrix of shape (n_samples, n_labels).
        test_size (float): Proportion of data to include in the test split.

    Returns:
        Tuple[np.ndarray, np.ndarray]: Indices for training and testing sets.
    """
    # Splitting the Multi-label dataset
    inv = 1 / test_size
    n_splits = max(2, int(round(inv)))  # Make sure it's at least 2
    stratifier = IterativeStratification(n_splits=n_splits, order=1)
    # Returning the train and test indices
    for train_idx, test_idx in stratifier.split(X, y):
        return train_idx, test_idx

## **Label Smoothing for Multi-label Targets**

Applies label smoothing to the binary ground-truth matrix. Instead of using hard 0/1 labels, the function softens them slightly to reduce overconfidence in model predictions and improve generalization. This is particularly useful in multi-label settings with imbalanced or noisy labels.

In [6]:
# applying the label smoothing
def apply_label_smoothing(y_true, smoothing=0.1):
    """
    Applies label smoothing to binary multi-label targets.

    Args:
        y_true (np.ndarray or torch.Tensor): Ground truth binary labels.
        smoothing (float): Smoothing factor to reduce label confidence.

    Returns:
        Same type as input: Smoothed labels.
    """
    return y_true * (1.0 - smoothing) + 0.5 * smoothing

## **Custom PyTorch Dataset for Antibiotic Resistance Spectra**

This class wraps MALDI-TOF spectral features and resistance labels into a PyTorch `Dataset` for use with data loaders.
It ensures:

* Features are 2D tensors with a channel dimension (e.g., for CNNs).
* Labels are cast to tensors and reshaped for compatibility (e.g., single-label vs. multi-label).

In [7]:
# Define a custom PyTorch dataset for antibiotic resistance classification
class AntibioticDataset(Dataset):
    """
    Custom PyTorch Dataset for antibiotic resistance classification using MALDI-TOF spectra.

    Args:
        X (np.ndarray or torch.Tensor): Input spectra of shape (N, C, L) or (N, L).
        y (np.ndarray or torch.Tensor): Binary multi-label targets of shape (N, num_labels).

    Returns:
        Tuple[torch.Tensor, torch.Tensor]: (sample, label) for each index.
    """
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1) if not torch.is_tensor(X) else X
        self.y = torch.tensor(y, dtype=torch.float32) if not torch.is_tensor(y) else y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        x = self.X[index]
        y = self.y[index]
        if len(y.shape) == 0:  # scalar label (when num_labels = 1)
            y = np.expand_dims(y, axis=0)  # make it shape (1,)
        return x, y


## **Prepare Year-wise Dataset Splits with Optional Label Smoothing**

This function takes in one or more dataframes containing MALDI-TOF mass spectra and resistance labels, then:

* Converts categorical resistance labels ('R', 'I', 'S') to binary values (1 for 'R' and 'I', 0 for 'S').
* Performs a multi-label stratified train/val/test split (70/10/20).
* Applies label smoothing if specified.
* Returns PyTorch `Dataset` objects for model training.

This modular setup ensures consistent, stratified splits and enables fair benchmarking.

In [8]:
from sklearn.preprocessing import Normalizer

def prepare_year_data(df_list, smooth=0, num_labels=5, split=True): 
    """
    Prepares train, validation, and test datasets from a dataframe or list of dataframes,
    or returns a complete dataset if split is False.

    Args:
        df_list (Union[pd.DataFrame, List[pd.DataFrame]]): Single dataframe or list of dataframes
            containing input features followed by label columns.
        smooth (float, optional): Label smoothing factor applied to training labels. Defaults to 0.
        num_labels (int, optional): Number of label columns at the end of the dataframe. Defaults to 5.
        split (bool, optional): Whether to split the data or not. Defaults to True.

    Returns:
        Tuple[AntibioticDataset, AntibioticDataset, AntibioticDataset] or AntibioticDataset:
            If split is True, returns a tuple of PyTorch Dataset objects:
                (train_dataset, val_dataset, test_dataset) split in a 70/10/20 ratio.
            If split is False, returns a single PyTorch Dataset object.
    """

    # Combine all dataframes if multiple are provided
    if isinstance(df_list, list):
        df = pd.concat(df_list, ignore_index=True)
    else:
        df = df_list
    
    # Split features and labels
    X = df.iloc[:, :-num_labels].values.astype(np.float32)  # Convert to float32
    y_raw = df.iloc[:, -num_labels:].values
    
    # Convert labels: 'R' and 'I' → 1, 'S' → 0
    y = ((y_raw == 'R') | (y_raw == 'I')).astype(np.float32)

    # No Split
    if not split:
        # Apply Max Normalization on the entire dataset
        normalizer = Normalizer(norm="max")
        X = normalizer.fit_transform(X)

        return AntibioticDataset(X, y)

    # Train/Val/Test Split
    # Perform 70/10/20 train/val/test multilabel split
    trainval_idx, test_idx = multilabel_train_test_split(X, y, test_size=0.2)
    X_trainval, y_trainval = X[trainval_idx], y[trainval_idx]
    train_idx, val_idx = multilabel_train_test_split(X_trainval, y_trainval, test_size=0.125)

    # Map back to global indices for val and train
    val_idx = trainval_idx[val_idx]
    train_idx = trainval_idx[train_idx]

    # Prepare final datasets
    X_train, y_train = X[train_idx], y[train_idx]
    y_train = apply_label_smoothing(y_train, smoothing=smooth)
    X_val, y_val = X[val_idx], y[val_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    # Apply Max Normalization (fit only on train)
    normalizer = Normalizer(norm="max")
    X_train = normalizer.fit_transform(X_train)
    X_val = normalizer.transform(X_val)
    X_test = normalizer.transform(X_test)

    # Wrap in PyTorch Dataset objects
    train_dataset = AntibioticDataset(X_train, y_train)
    val_dataset = AntibioticDataset(X_val, y_val)
    test_dataset = AntibioticDataset(X_test, y_test)

    return train_dataset, val_dataset, test_dataset

## **Multi-Head Convolutional Neural Network (1D)**

This CNN-based architecture is designed for multi-label classification from 1D input signals (e.g., time series or genomic profiles). It consists of four convolutional blocks with residual connections, followed by fully connected layers. The final representation is passed through multiple independent output heads, each predicting the probability of a binary class (e.g., resistance for each antibiotic).

A skip connection from early features is fused into the final convolutional block to retain low-level spatial information. This design improves gradient flow and representation capacity across layers.

Each output head uses a sigmoid activation to enable independent label prediction, suitable for multi-label tasks with highly imbalanced class distributions.


In [9]:
class MultiHeadCNN(nn.Module):
    """
    A multi-head convolutional neural network for multi-label classification 
    of MALDI-TOF spectra. Each output head predicts one antibiotic label.

    Args:
        num_labels (int): Number of labels to predict (i.e., antibiotics).
        input_channels (int): Number of input channels (typically 1 for mass spectra).
        dropout (float): Dropout rate for regularization.
        input_length (int): Length of the input 1D spectrum.

    Returns:
        torch.Tensor: Predictions of shape (batch_size, num_labels), with sigmoid outputs.
    """
    def __init__(self, num_labels: int, input_channels: int = 1, dropout=0.65, input_length: int = 6000):
        super(MultiHeadCNN, self).__init__()
        self.num_labels = num_labels

        # Conv Block 1
        self.conv1 = nn.Conv1d(input_channels, 128, kernel_size=17, padding=8)
        self.bn1 = nn.BatchNorm1d(128)

        # Conv Block 2
        self.conv2 = nn.Conv1d(128, 256, kernel_size=9, padding=4)
        self.bn2 = nn.BatchNorm1d(256)
        self.shortcut2 = nn.Conv1d(128, 256, kernel_size=1)

        # Conv Block 3
        self.conv3 = nn.Conv1d(256, 512, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(512)
        self.shortcut3 = nn.Conv1d(256, 512, kernel_size=1)

        # Conv Block 4
        self.conv4 = nn.Conv1d(512, 512, kernel_size=5, padding=2)
        self.bn4 = nn.BatchNorm1d(512)
        self.shortcut4 = nn.Identity()

        # Deterministic Downsampler (Conv1d with stride)
        self.downsample_long_skip = nn.Conv1d(128, 512, kernel_size=3, stride=4, padding=1)

        # Pooling
        self.pool = nn.MaxPool1d(kernel_size=2)  # deterministic

        # Output size after 4 poolings
        conv_out_length = input_length // (2 ** 4)
        self.flatten_dim = 512 * conv_out_length

        self.fc1 = nn.Linear(self.flatten_dim, 512)
        self.dropout1 = nn.Dropout(dropout)

        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 128)

        self.heads = nn.ModuleList([nn.Linear(128, 1) for _ in range(num_labels)])

    def forward(self, x):
        # Block 1
        x1 = F.relu(self.bn1(self.conv1(x)))
        x1_p = self.pool(x1)

        # Block 2
        x2 = F.relu(self.bn2(self.conv2(x1_p) + self.shortcut2(x1_p)))
        x2_p = self.pool(x2)

        # Block 3
        x3 = F.relu(self.bn3(self.conv3(x2_p) + self.shortcut3(x2_p)))
        x3_p = self.pool(x3)

        # Deterministic downsampling for long skip
        long_skip_feat = self.downsample_long_skip(x1_p)

        # Block 4
        x4 = F.relu(self.bn4(self.conv4(x3_p) + self.shortcut4(x3_p) + long_skip_feat))
        x4_p = self.pool(x4)

        # Flatten + FC
        x_flat = x4_p.view(x4_p.size(0), -1)
        x = self.dropout1(F.relu(self.fc1(x_flat)))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))

        outputs = [torch.sigmoid(head(x)) for head in self.heads]
        return torch.cat(outputs, dim=1)


## **Function for computing shap values**

In [10]:
def save_shap_means_to_json(shap_dict, filename="shap_summary.json"):
    """
    Saves SHAP summary values into a JSON file.
    Ensures NumPy arrays are converted to standard Python lists for compatibility.

    Args:
        shap_dict (dict): Dictionary of SHAP means for each label.
        filename (str): Output JSON filename.
    """
    def convert_ndarray(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: convert_ndarray(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_ndarray(i) for i in obj]
        else:
            return obj

    safe_dict = convert_ndarray(shap_dict)

    with open(filename, 'w') as f:
        json.dump(safe_dict, f, indent=2)


def compute_shap_summary(trained_model, test_dataset, num_labels=5, background_size=5, n_samples=100, file = "shap_summary.json"):
    """
    Computes SHAP values for representative samples in a multi-label classification task.
    Separates SHAP importance by resistance status (resistant/susceptible) and saves the means per label.

    Args:
        trained_model (torch.nn.Module): Trained model with multi-label output.
        test_dataset (torch.utils.data.Dataset): Test dataset to explain.
        num_labels (int): Number of output labels/classes.
        background_size (int): Number of background samples for SHAP GradientExplainer.
        n_samples (int): Number of samples to use for SHAP explanations.
    """

    # Set model to evaluation mode and move to CPU for SHAP
    trained_model.eval()
    model_cpu = trained_model.to('cpu')

    # Extract all data from the test dataset
    all_x, all_y = [], []
    for xb, yb in DataLoader(test_dataset, batch_size=512):
        all_x.append(xb)
        all_y.append(yb)
    X = torch.cat(all_x).squeeze(1).numpy()  # Features (N, F)
    Y = torch.cat(all_y).numpy()            # Labels (N, L)

    # Use stratified sampling to select a representative subset of samples
    stratifier = IterativeStratification(
        n_splits=2,
        order=1,
        sample_distribution_per_fold=[n_samples / len(X), 1 - (n_samples / len(X))]
    )
    _, selected_idx = next(stratifier.split(X, Y))
    X_subset = torch.tensor(X[selected_idx]).unsqueeze(1).float()  # Shape: (n_samples, 1, F)
    Y_subset = torch.tensor(Y[selected_idx]).float()               # Shape: (n_samples, L)

    # Select a small number of background samples required for SHAP
    bg_idx = np.random.choice(len(X_subset), background_size, replace=False)
    X_bg = X_subset[bg_idx]

    # Step 4: Initialize SHAP GradientExplainer
    explainer = shap.GradientExplainer(model_cpu, X_bg)

    # Initialize containers for SHAP values per label
    num_features = X_subset.shape[-1]
    shap_values_resistant = [[] for _ in range(num_labels)]
    shap_values_susceptible = [[] for _ in range(num_labels)]

    # Loop over selected samples to compute SHAP values
    for i in tqdm(range(len(X_subset))):
        x_sample = X_subset[i:i+1]
        y_sample = Y_subset[i]

        shap_vals = explainer.shap_values(x_sample)  # List of (1, F) arrays

        for label_index in range(num_labels):
            shap_for_label = shap_vals[label_index][0].squeeze()
            label_value = y_sample[label_index].item()

            # Append to respective group based on label
            if label_value == 1:
                shap_values_resistant[label_index].append(shap_for_label)
            else:
                shap_values_susceptible[label_index].append(shap_for_label)

    # Compute mean SHAP values and print top features
    shap_summary = {}

    for label_index in range(num_labels):
        print(f"\n===== LABEL {label_index} =====")
        r_vals = shap_values_resistant[label_index]
        s_vals = shap_values_susceptible[label_index]

        r_mean = np.mean(r_vals, axis=0) if r_vals else np.zeros(num_features)
        s_mean = np.mean(s_vals, axis=0) if s_vals else np.zeros(num_features)

        shap_summary[f"label_{label_index}"] = {
            "resistant_mean": r_mean,
            "susceptible_mean": s_mean
        }

        print("Top 10 RESISTANT features:")
        for idx in np.argsort(-np.abs(r_mean))[:10]:
            print(f"  mz_{idx}: {r_mean[idx]:.4f}")

        print("Top 10 SUSCEPTIBLE features:")
        for idx in np.argsort(-np.abs(s_mean))[:10]:
            print(f"  mz_{idx}: {s_mean[idx]:.4f}")

    save_shap_means_to_json(shap_summary, file)
    print("Saved SHAP summary to shap_summary.json")

# **Experiment pipeline**

We have defined all the utilities for this experiment, now we conduct the experiement

## **Loading Processed DRIAMS-A Datasets for *Staphylococcus aureus***

We use the `generate_driam_sets` utility to load mass spectrometry datasets for *Staphylococcus aureus* from the DRIAMS-A collection for the years 2015 to 2018. Each dataset includes binarized resistance labels for the following antibiotics:

- Ciprofloxacin
- Fusidic acid
- Oxacillin
- Ceftriaxone
- Clindamycin


In [11]:
# Define target antibiotics for resistance profiling
antibiotics = ['Ciprofloxacin', 'Fusidic acid', 'Oxacillin', 'Ceftriaxone', 'Clindamycin']
# Set target species
specie = 'Staphylococcus aureus'

# Load binarized, binned DRIAMS-A datasets for S. aureus from 2015 to 2018
Saureus_A_2015 = generate_driam_sets(specie, antibiotics, '/kaggle/input/driams-processed-set/DriamsA_2015_bin3_2000-20000.csv')
Saureus_A_2016 = generate_driam_sets(specie, antibiotics, '/kaggle/input/driams-processed-set/DriamsA_2016_bin3_2000-20000.csv')
Saureus_A_2017 = generate_driam_sets(specie, antibiotics, '/kaggle/input/driams-processed-set/DriamsA_2017_bin3_2000-20000.csv')
Saureus_A_2018 = generate_driam_sets(specie, antibiotics, '/kaggle/input/driams-processed-set/DriamsA_2018_bin3_2000-20000.csv')

Processing file at /kaggle/input/driams-processed-set/DriamsA_2015_bin3_2000-20000.csv

Processing species: Staphylococcus aureus in file: /kaggle/input/driams-processed-set/DriamsA_2015_bin3_2000-20000.csv
Processing file at /kaggle/input/driams-processed-set/DriamsA_2016_bin3_2000-20000.csv



A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
Columns (6071) have mixed types. Specify dtype option on import or set low_memory=False.



Processing species: Staphylococcus aureus in file: /kaggle/input/driams-processed-set/DriamsA_2016_bin3_2000-20000.csv



A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Processing file at /kaggle/input/driams-processed-set/DriamsA_2017_bin3_2000-20000.csv


Columns (6071) have mixed types. Specify dtype option on import or set low_memory=False.



Processing species: Staphylococcus aureus in file: /kaggle/input/driams-processed-set/DriamsA_2017_bin3_2000-20000.csv



A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Processing file at /kaggle/input/driams-processed-set/DriamsA_2018_bin3_2000-20000.csv

Processing species: Staphylococcus aureus in file: /kaggle/input/driams-processed-set/DriamsA_2018_bin3_2000-20000.csv



A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


## **Combining and Shuffling S.aureus Datasets (2015–2018)**

We concatenate the four yearly datasets into a single DataFrame to form a unified dataset for *Staphylococcus aureus*. The rows are then shuffled using a fixed random seed (`SEED`) to ensure reproducibility while maintaining class balance and feature alignment.


In [12]:
# Combine in fixed order
combined_data = pd.concat(
    [Saureus_A_2015, Saureus_A_2016, Saureus_A_2017, Saureus_A_2018],
    ignore_index=True
)

# Shuffle the combined dataset with a fixed seed for reproducibility
combined_data = combined_data.sample(frac=1, random_state=SEED).reset_index(drop=True)

## **Preparing train, val and test split and analyzing distribution**
once the datasets are combined and shuffled, we split them into train, val and test set using `prepare_year_data` utility in the ratio of 70/10/20. we apply label smoothing of 0.05. Next we analyze class distriubtion using `analyze_pytorch_dataset`


In [13]:
train_dataset, val_dataset, test_dataset = prepare_year_data(combined_data, smooth=0.05, num_labels=5)

## **Training the model for saureus**


In [14]:
# Initialize model architecture
device = "cuda" if torch.cuda.is_available() else "cpu"
sa_model = MultiHeadCNN(num_labels=5, dropout = 0).to(device)

#load the pre-trained model weights
sa_model.load_state_dict(torch.load("/kaggle/input/saureus-2015-2018-detmodel-seta/saureus_best_model.pt"))


#move the model to cuda
sa_model = sa_model.to(device)

## **Compute shap for saureus**

In [15]:
compute_shap_summary(sa_model, test_dataset, num_labels=5, background_size=5, n_samples=100, file = "shap_summary_sa.json")

100%|██████████| 100/100 [3:49:07<00:00, 137.47s/it]


===== LABEL 0 =====
Top 10 RESISTANT features:
  mz_1017: 0.0511
  mz_1001: 0.0459
  mz_473: 0.0448
  mz_81: 0.0441
  mz_1011: 0.0424
  mz_2625: 0.0396
  mz_80: 0.0381
  mz_139: 0.0363
  mz_136: -0.0357
  mz_1806: 0.0351
Top 10 SUSCEPTIBLE features:
  mz_137: -0.0613
  mz_1010: -0.0600
  mz_81: 0.0528
  mz_473: 0.0497
  mz_136: -0.0474
  mz_2625: 0.0462
  mz_135: -0.0429
  mz_1017: 0.0410
  mz_139: 0.0395
  mz_424: 0.0391

===== LABEL 1 =====
Top 10 RESISTANT features:
  mz_1807: -0.0251
  mz_1808: -0.0239
  mz_1175: 0.0225
  mz_81: -0.0218
  mz_681: -0.0205
  mz_1176: -0.0156
  mz_80: -0.0155
  mz_1809: 0.0154
  mz_1811: -0.0150
  mz_352: 0.0148
Top 10 SUSCEPTIBLE features:
  mz_80: -0.0156
  mz_681: -0.0144
  mz_1145: -0.0122
  mz_81: -0.0116
  mz_1337: -0.0108
  mz_1336: -0.0106
  mz_545: -0.0106
  mz_425: -0.0105
  mz_22: -0.0104
  mz_1144: -0.0102

===== LABEL 2 =====
Top 10 RESISTANT features:
  mz_81: 0.1745
  mz_135: -0.1143
  mz_136: -0.1128
  mz_139: 0.1087
  mz_45: -0.1005
